<a href="https://colab.research.google.com/github/bipasnadulal/aws-future-ai-programmer-nanodegree/blob/main/implementing_attention_block.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
from pathlib import Path

text = Path("/content/tiny-shakespeare.txt").read_text()

In [6]:
print(text[0:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return "".join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [8]:
tokenizer = CharTokenizer.train_from_text(text)

In [9]:
print(tokenizer.encode("Hello World"))

tensor([20, 43, 50, 50, 53,  1, 35, 53, 56, 50, 42])


In [10]:
print(tokenizer.decode(tokenizer.encode("Hello World")))

Hello World


In [11]:
print(f"Vocabulary size: {tokenizer.vocabulary_size()}")

Vocabulary size: 65


In [12]:
from torch.utils.data import Dataset

class TokenIdsDataset(Dataset):
  def __init__(self, data, block_size):
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, pos):
    assert pos < len(self.data) - self.block_size

    x = self.data[pos : pos + self.block_size]
    y = self.data[pos + 1 : pos + self.block_size + 1]

    return x, y

### Model Configuration:
- Vocabulary Size: The number of unique token IDs supported by the tokenizer.
- Context Size: The maximum number of tokens the model can see at once.
- Embedding Dimension: The size of the embedding vectors.
- Number of Heads: The number of attention heads, each processing input independently.
- Number of Layers: The number of Transformer blocks in the model.
- Dropout Rate: The proportion of outputs set to zero in Dropout layers.
- Use Bias: A boolean indicating whether the linear transformations should include bias terms.

In [13]:
config = {
    "vocabulary_size": tokenizer.vocabulary_size,
    "context_size":256,
    "embedding_dim":768,
    "heads_num":12,
    "layers_num":10,
    "dropout_rate":0.1,
    "use_bias":False
}

config['head_size'] = config['embedding_dim'] // config['heads_num']

### Attention Head implementation

- Attention Head Class
  - constructor
      - Q, K, V Matrices: Used to convert input embedding vectors into Q, K and V matrices.
      - Dropout Layer: For regularization (to prevent overfitting)
      - Casual Attention Mask: Ensures the model cannot "cheat" by looking at future tokens.

In [14]:
class AttentionHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.Q_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.K_weights = nn.Linear(
        config['embedding_dim'], config['head-size'], config['use_bias']
    )
    self.V_weights = nn.Linear(
        config['embedding_dim'], config['head-size'], config['use_bias']
    )
    self.dropout = nn.Dropout(config['dropout_rate'])

    casual_attention_mask = torch.tril(
        torch.ones(config['context_size'], config['context_size'])
    )
    self.register_buffer('casual_attention_mask', casual_attention_mask)

  def forward(self, input):
    batch_size, tokens_num, embedding_dim = input.shape
    Q = self.Q_weights(input)
    K = self.K_weights(input)
    V = self.V_weights(input)

    attention_scores = Q @ K.transpose(1, 2)
    attention_scores = attention_scores.masked_fill(
        self.casual_attention_mask[:tokens_num, :tokens_num] == 0,
        -torch.inf
    )
    attention_scores = attention_scores / (K.shape[-1] ** 0.5)
    attention_scores = torch.softmax(attention_scores, dim=-1)
    attention_scores = self.dropout(attention_scores)

    return attention_scores @ V

### Testing the Attenton Head

- Input tensor: A random input tensor is used to check the implementation.

- Output: Verify that the output has the correct dimensions.

In [15]:
class AttentionHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.Q_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.K_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.V_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.dropout = nn.Dropout(config['dropout_rate'])

    casual_attention_mask = torch.tril(
        torch.ones(config['context_size'], config['context_size'])
    )
    self.register_buffer('casual_attention_mask', casual_attention_mask)

  def forward(self, input):
    batch_size, tokens_num, embedding_dim = input.shape
    Q = self.Q_weights(input)
    K = self.K_weights(input)
    V = self.V_weights(input)

    attention_scores = Q @ K.transpose(1, 2)
    attention_scores = attention_scores.masked_fill(
        self.casual_attention_mask[:tokens_num, :tokens_num] == 0,
        -torch.inf
    )
    attention_scores = attention_scores / (K.shape[-1] ** 0.5)
    attention_scores = torch.softmax(attention_scores, dim=-1)
    attention_scores = self.dropout(attention_scores)

    return attention_scores @ V

input = torch.rand(8, config['context_size'], config['embedding_dim'])
ah = AttentionHead(config)

output = ah(input)
print(output.shape)

torch.Size([8, 256, 64])


### Multi-Head Attention Implementation

- Inherit from nn.Module. (nn.Module is base class for all neural network modules in PyTorch)
- Create Multiple Heads: Instantiate multiple AttentionHead instances and store them in a ModuleList.
- Additional Linear Layer: Applies to each embedding vector.
- Dropout Layer: For regularization.

In [16]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()

    heads_list = [AttentionHead(config) for _ in range(config['heads_num'])]
    self.heads = nn.ModuleList(heads_list)
    self.linear = nn.Linear(config['embedding_dim'], config['embedding_dim'])
    self.dropout = nn.Dropout(config['dropout_rate'])

  def forward(self, input):
    heads_outputs = [head(input) for head in self.heads]
    scores_change = torch.cat(heads_outputs, dim=-1)

    scores_change = self.linear(scores_change)

    return self.dropout(scores_change)

In [17]:
mha = MultiHeadAttention(config)
output = mha(input)
print(output.shape)

torch.Size([8, 256, 768])
